这是**数据包络分析 (Data Envelopment Analysis, DEA)** 的详细解析。

DEA是评价**“投入产出效率”**的神器。
如果你手头有一堆机构（如分公司、医院、学校），你有它们的**投入数据**（花了多少钱、用了多少人）和**产出数据**（赚了多少钱、治好了多少人），想知道谁干得好、谁在摸鱼，一定要用DEA。

---

### 一、 算法原理
**核心思想**：**“不以偏概全，让每个人都展示最强的一面。”**

1.  **多投入多产出**：传统的评价通常是 单产出/单投入（比如：人均GDP）。但现实是复杂的，比如医院，投入有医生人数、设备数量，产出有治愈人数、病床周转率。DEA能同时处理这些。
2.  **相对效率**：DEA不跟标准答案比，而是**跟同行比**。
    *   它会在所有样本中画出一条**“生产前沿面”**（最好的那波人组成的曲线）。
    *   落在前沿面上的，效率就是 1（有效）。
    *   落在里面的，效率就小于 1（无效），离前沿面越远越差。
3.  **权重自适应**：DEA最牛的地方在于**不需要你给权重**。它允许每个决策单元（DMU）自己安排最有利于自己的权重组合。如果连你自己选的最优权重都干不过别人，那你就是真的菜。

---

### 二、 推导与步骤 (CCR模型)

最经典的是 **CCR模型**（假设规模报酬不变）。

假设有 $n$ 个评价对象（DMU），每个对象有 $m$ 种投入 $X$， $s$ 种产出 $Y$。
我们要评价第 $k$ 个对象的效率。

#### 1. 原始规划 (分式规划)
目标是最大化效率值 $\theta$：
$$ \max \theta_k = \frac{\sum_{r=1}^{s} u_r y_{rk}}{\sum_{i=1}^{m} v_i x_{ik}} $$
约束条件是：对于所有的 $j=1...n$，其效率值都不能超过1。
$$ \frac{\sum_{r=1}^{s} u_r y_{rj}}{\sum_{i=1}^{m} v_i x_{ij}} \le 1 $$
其中 $u, v \ge 0$ 是权重。

#### 2. 转化为线性规划 (对偶模型)
分式规划很难解，我们通常将其转化为**对偶形式的线性规划**来求解。这也是代码中实际运行的公式。

对于第 $k$ 个 DMU，我们求解最小值 $\theta$（效率值）：
$$ \min \theta $$
约束条件：
1.  **投入约束**：别人的加权投入不能比我打折后还少
    $$ \sum_{j=1}^{n} \lambda_j x_{ij} \le \theta x_{ik} $$
2.  **产出约束**：别人的加权产出必须比我多（或持平）
    $$ \sum_{j=1}^{n} \lambda_j y_{rj} \ge y_{rk} $$
3.  **变量范围**：$\lambda_j \ge 0$

*   如果算出来 $\theta = 1$，说明你是有效的（在前沿面上）。
*   如果 $\theta = 0.8$，说明你只需用当前 80% 的投入就能达到现在的产出（也就是说你浪费了 20%）。

---

### 三、 适用场景
1.  **多投入、多产出**：这是硬性要求。
2.  **同类型单位比较**：评价30个省份的医疗效率、评价10家物流公司的运输效率、评价50所大学的科研绩效。
3.  **无需主观权重**：当你不知道“医生人数”和“设备价格”哪个更重要时，用DEA。

---

### 四、 Python 代码框架

Python没有像SPSS那样傻瓜式的DEA库（有一个 `pyDEA` 但很久没更新了）。
最稳健的方法是利用 `scipy.optimize.linprog` 手写一个类。我已经帮你封装好了，直接复制即用。

这里实现的是最常用的 **CCR模型 (投入导向)**。

```python
import numpy as np
import pandas as pd
from scipy.optimize import linprog

class DEA:
    def __init__(self, inputs, outputs):
        """
        初始化 DEA 模型
        :param inputs: pandas DataFrame or np.array, 投入数据 (n_samples x m_inputs)
        :param outputs: pandas DataFrame or np.array, 产出数据 (n_samples x s_outputs)
        """
        self.inputs = np.array(inputs)
        self.outputs = np.array(outputs)
        self.n = self.inputs.shape[0] # DMU数量
        self.m = self.inputs.shape[1] # 投入指标数量
        self.s = self.outputs.shape[1] # 产出指标数量
        
        # 结果容器
        self.efficiency = []
        
    def fit(self):
        """
        使用 CCR 模型 (投入导向) 计算每个 DMU 的效率
        """
        # 对每一个 DMU (k) 进行循环求解
        for k in range(self.n):
            # === 线性规划配置 ===
            # 目标函数: min theta
            # 决策变量向量构造: [theta, lambda_1, lambda_2, ..., lambda_n]
            # 所以总变量个数 = 1 + n
            
            # 1. 目标函数系数 c
            # 我们只优化 theta (第一个变量), lambda 的系数都是 0
            c = np.zeros(1 + self.n)
            c[0] = 1 
            
            # 2. 不等式约束 A_ub * x <= b_ub
            # 我们有 m 个投入约束 和 s 个产出约束
            
            # (1) 投入约束: sum(lambda * x_j) <= theta * x_k
            # 移项得: sum(lambda * x_j) - theta * x_k <= 0
            # 对应系数: [-x_ik, x_i1, x_i2, ..., x_in]
            A_inputs = []
            for i in range(self.m):
                row = [-self.inputs[k, i]] + list(self.inputs[:, i])
                A_inputs.append(row)
            
            # (2) 产出约束: sum(lambda * y_j) >= y_k
            # scipy 只支持 <=, 所以两边取反: -sum(lambda * y_j) <= -y_k
            # 对应系数: [0, -y_r1, -y_r2, ..., -y_rn] (theta的系数是0)
            A_outputs = []
            for r in range(self.s):
                row = [0] + list(-self.outputs[:, r])
                A_outputs.append(row)
            
            # 合并约束矩阵
            A_ub = np.array(A_inputs + A_outputs)
            
            # 构造 b_ub
            # 投入约束右边是 0
            # 产出约束右边是 -y_rk
            b_ub = np.concatenate([np.zeros(self.m), -self.outputs[k, :]])
            
            # 3. 变量边界
            # theta >= 0 (其实理论上没有上限，但效率一般<=1，不过不设限让模型自己跑)
            # lambda >= 0
            bounds = [(0, None)] + [(0, None)] * self.n
            
            # === 求解 ===
            res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
            
            if res.success:
                self.efficiency.append(res.x[0]) # res.x[0] 就是 theta
            else:
                self.efficiency.append(np.nan)
                print(f"DMU {k} 求解失败")
        
        return np.array(self.efficiency)

    def analysis(self, index_names=None):
        """
        输出分析结果表格
        """
        eff = np.array(self.efficiency)
        
        # 简单的评价逻辑
        results = []
        for e in eff:
            if np.isclose(e, 1.0):
                results.append("DEA有效")
            else:
                results.append("非DEA有效")
        
        df_res = pd.DataFrame({
            '效率值(Theta)': np.round(eff, 4),
            '评价结果': results
        })
        
        if index_names is not None:
            df_res.index = index_names
            
        return df_res

# ================= 使用示例 =================

if __name__ == "__main__":
    # 场景：评价 5 家医院的运营效率
    # 投入 (越少越好)：医生人数、护士人数、运营成本(万)
    inputs_data = pd.DataFrame({
        '医生': [20, 50, 40, 60, 25],
        '护士': [50, 100, 90, 120, 60],
        '成本': [100, 300, 250, 400, 120]
    })
    
    # 产出 (越多越好)：门诊量(人次)、治愈人数、病床周转率
    outputs_data = pd.DataFrame({
        '门诊': [2000, 6000, 4500, 6500, 2800],
        '治愈': [180, 500, 400, 550, 220],
        '周转': [80, 90, 85, 95, 88]
    })
    
    print("投入数据:\n", inputs_data)
    print("产出数据:\n", outputs_data)
    
    # 1. 初始化模型
    dea = DEA(inputs_data, outputs_data)
    
    # 2. 计算效率
    dea.fit()
    
    # 3. 输出结果
    hospital_names = ['医院A', '医院B', '医院C', '医院D', '医院E']
    result_df = dea.analysis(index_names=hospital_names)
    
    print("-" * 30)
    print("DEA 评价结果:")
    print(result_df)
    
    # 结果解读：
    # 效率值 = 1.0000 -> 它是标杆，投入产出比非常棒。
    # 效率值 = 0.8500 -> 它是非有效的，说明它当前的产出，理论上只需要投入现在的 85% 就能做到（或者在当前投入下，产出还能更高）。
```

### 代码使用的“修修补补”指南：

1.  **数据方向问题**：
    *   DEA非常严格：**投入**必须是投入（比如成本、人数），**产出**必须是产出（比如利润、产量）。
    *   *如果有负向产出怎么办？*（比如“废品率”，越低越好，但它是产出）。
        *   **方法一（倒数法）**：取倒数，变成 $1/x$。
        *   **方法二（大数减小数）**：用一个足够大的数 $M$ 减去它，变成正向指标。
        *   **千万不要直接放进去**，否则线性规划会算出奇怪的结果。

2.  **效率值全都是 1？**
    *   如果你发现算出来大家都是1，或者绝大多数是1。
    *   **原因**：你的指标太多，样本太少！
    *   **经验法则**：样本数量（DMU数）最好大于 $2 \times (投入指标数 + 产出指标数)$。比如你有3个投入、2个产出，那你至少得有10个样本。否则大家的个性能被无限放大，每个人都是第一名。

3.  **CCR vs BCC**：
    *   上面的代码是 **CCR模型**（假设规模报酬不变，Constant Returns to Scale）。这最常用，评价的是“综合技术效率”。
    *   还有一种叫 **BCC模型**（假设规模报酬可变），它算出来的是“纯技术效率”。
    *   *怎么改代码变BCC？* 只需要在 `A_ub` 矩阵里加一行约束：$\sum \lambda = 1$。如果你需要BCC，在构造 `A_ub` 的时候加上这行等式约束即可。但在大多数基础建模比赛中，CCR足够说明问题。

4.  **结果怎么写进论文？**
    *   通常不仅要列出效率值，还要分析**“投影值”**（Slack）。
    *   虽然上面的代码只输出了 $\theta$，但其实你可以分析：对于非有效的医院，应该减少多少医生（投入冗余），增加多少病人（产出不足）。这部分也就是常说的“非DEA有效单元的改进意见”。